# Phase 2: Preprocessing and Reproducible Splitting

This notebook converts the verified MiraBest Zenodo v1 basic binary dataset to normalized float tensors and creates stratified train, validation, and test splits. Raw files are not modified, and augmentation is applied only to the training view.

In [ ]:
from pathlib import Path
import sys
from collections import Counter

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.download_mirabest import load_mirabest
from src.data.split_dataset import prepare_splits

dataset = load_mirabest(PROJECT_ROOT / 'data' / 'raw' / 'mirabest', download=False)
print(f'Loaded samples: {len(dataset)}')

In [ ]:
split_indices, preprocessed_splits = prepare_splits(
    dataset,
    seed=42,
    output_dir=PROJECT_ROOT / 'data' / 'splits',
    figure_path=PROJECT_ROOT / 'outputs' / 'figures' / 'split_class_distribution.png',
)

for split_name, indices in split_indices.items():
    counts = Counter(int(dataset[index][1]) for index in indices)
    print(split_name, len(indices), dict(counts))

The split counts and class counts above are calculated from the loaded dataset. The saved indices make the assignment reproducible with seed 42. The training view uses conservative random rotations/flips; validation and test views use only deterministic tensor conversion and normalization.

In [ ]:
for split_name, split_dataset in preprocessed_splits.items():
    image, label = split_dataset[0]
    print(split_name, tuple(image.shape), image.dtype, label, float(image.min()), float(image.max()))